# 01 · Data prep — smart-turn v3.2 → English + Hindi + a multilingual tail

**Kaggle settings:** CPU (no GPU needed) · **Internet ON** · takes ~2-3 h.

Streams `pipecat-ai/smart-turn-data-v3.2-train` (41 GB) and `-test`, resamples
to 16 kHz mono, keeps the last 8 s, and writes FLAC into 64 **ZIP shards**
(`/kaggle/working/prep/shards/shard_NN.zip`) next to `manifest.parquet`.

**Why shards:** Kaggle's kernel-output publishing silently fails past roughly
100k files — the version completes, but its output is an 845-byte empty
`_output_.zip`. The earlier 56k-clip prep published fine; this one keeps ~122k,
so the clips ride inside stored (uncompressed — FLAC already is) zips. Each
entry's arcname is exactly the `path` the manifest records
(`audio/<2-hex>/<id>.flac`), so notebook 02 extracts the shards to
`/tmp/prep_audio` and uses that as the audio root, unchanged from the old
loose-file layout.

**Composition** (this is the E6 "full data" prep, a superset of the E1-E4 one):

| stream | kept | cap |
|---|---|---|
| train, `eng` | all (effectively uncapped) | 33,000 / label |
| train, `hin` | all | — |
| train, everything else | a tail for multilingual robustness | 850 / (language, label) |
| test | `eng` + `hin` only | — |

English/Hindi are renamed to `english`/`hindi` (every downstream consumer masks
on the long names); other languages keep their **raw ISO-639-3 code** and are
all assigned `split="train"` — validation stays EN+HI so that best-checkpoint
selection is comparable with the earlier experiments, and the test stream is
untouched so the headline numbers keep meaning the same thing.

**Size:** expect **~16-17 GB**; Kaggle's `/kaggle/working` limit is ~19.6 GB.
The final cell prints the working-size total — that printout **is** the guard:
if it comes out near 19 GB, lower `OTHER_CAP_PER_LANG_LABEL` and re-run rather
than pushing a training job at a truncated dataset.

**When it finishes:** *Save Version* → after it completes, create a Kaggle
Dataset from this notebook's output named **`smart-turn-enhi-prep`**
(New Dataset → import from notebook output). Training runs attach that dataset.

**Interrupted?** A committed batch run starts from an empty `/kaggle/working`,
so there is nothing to resume from — just *Save & Run All* again. Prep normally
finishes well inside a single session. (The resume bookkeeping in the code only
helps when you re-run cells inside one live interactive session — and a shard
whose writer was killed outright loses its central directory, which the final
cell's `entries >= manifest rows` assert catches. Re-run clean if it trips.)


In [ ]:
%pip install -q "datasets>=3.6,<4" soundfile polars

In [ ]:
import hashlib, io, json, time, zipfile
from pathlib import Path

import numpy as np
import polars as pl
import soundfile as sf
from datasets import Audio, load_dataset

OUT = Path("/kaggle/working/prep")
SHARD_DIR = OUT / "shards"
SHARD_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST = OUT / "manifest.jsonl"

SR = 16000
MAX_SECONDS = 8.0
EN_TRAIN_CAP_PER_LABEL = 33000   # english train rows per label (~all of them)
OTHER_CAP_PER_LANG_LABEL = 850   # non-EN/HI train rows per (language, label)
VAL_PCT = 5                      # % of train-source rows held out as val
# Audio goes into 64 ZIPs rather than loose files: Kaggle's kernel-output
# publishing silently fails somewhere past ~100k files -- the version "succeeds"
# with an 845-byte empty _output_.zip and no error. The 56k-clip prep published
# fine; this one keeps ~122k, so the clips ride in shard zips and notebook 02
# extracts them to /tmp before training.
SHARDS = 64

# smart-turn v3.2 tags language with ISO-639-3 codes; the manifest (and every
# downstream consumer) uses the long names for the two languages we report on,
# so normalise those once here and key EVERYTHING below -- rec, counters,
# resume rebuild -- on the normalised value. Every other language keeps its raw
# ISO code, which is what train.py's "multilingual_other" slice masks against.
LANG_MAP = {"eng": "english", "hin": "hindi"}
CORE = ("eng", "hin")

# a killed session can leave a half-written last line; drop it before appending
if MANIFEST.exists() and MANIFEST.stat().st_size:
    with open(MANIFEST, "rb+") as f:
        f.seek(-1, 2)
        if f.read(1) != b"\n":
            cut = MANIFEST.read_bytes().rfind(b"\n")
            f.truncate(cut + 1)
            print(f"truncated partial last line (kept {cut + 1} bytes)")

# ---- resume: rebuild done-set and counters from an existing manifest ----
done_ids, counts = set(), {}
if MANIFEST.exists():
    for line in open(MANIFEST):
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            continue
        done_ids.add(r["id"])
        src = "test" if r["split"] == "test" else "train"
        counts[(r["language"], r["label"], src)] = \
            counts.get((r["language"], r["label"], src), 0) + 1
print(f"resuming with {len(done_ids)} rows, counts={counts}")

mf = open(MANIFEST, "a")

# ---- shard zips: 64 open handles, opened lazily, closed in the finally ----
zips = {}

def shard_of(rid):
    # md5 rather than rid[:2] straight: ids are uuid strings today, but hashing
    # keeps the split uniform (and defined) whatever an id happens to look like.
    return int(hashlib.md5(rid.encode()).hexdigest()[:2], 16) % SHARDS

def shard_zip(i):
    zf = zips.get(i)
    if zf is None:
        # mode "a" creates the shard, and on a re-run inside a live session
        # appends to the existing one. Caveat: ZipFile writes the central
        # directory at close(), so a hard kill leaves the last-touched shards
        # without one -- "a" then starts a fresh archive at the end of the file
        # and the earlier entries stop being listed. The finalize cell's
        # "entries >= manifest rows" assert is what catches that; the fix is a
        # clean re-run, not a resume.
        zf = zipfile.ZipFile(SHARD_DIR / f"shard_{i:02d}.zip", mode="a",
                             compression=zipfile.ZIP_STORED)  # FLAC: already compressed
        zips[i] = zf
    return zf

def process(row, src):
    # a null language tag would otherwise write "language": null into the
    # manifest and break every downstream string mask
    code = row["language"] or "unk"
    core = code in CORE
    # the test stream stays EN+HI: overall/test numbers must keep comparing
    # like with like across E1-E6.
    if src == "test" and not core:
        return 0
    lang = LANG_MAP.get(code, code)
    label = int(bool(row["endpoint_bool"]))
    rid = row["id"]
    if rid in done_ids:
        return 0
    if src == "train":
        if lang == "english" and \
                counts.get(("english", label, "train"), 0) >= EN_TRAIN_CAP_PER_LABEL:
            return 0
        if not core and \
                counts.get((lang, label, "train"), 0) >= OTHER_CAP_PER_LANG_LABEL:
            return 0
    wav = np.asarray(row["audio"]["array"], dtype=np.float32)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = wav[-int(MAX_SECONDS * SR):]
    if len(wav) < int(0.3 * SR):
        return 0
    # arcname == the manifest's "path", so extracting a shard reproduces the
    # exact audio/<2-hex>/<id>.flac tree the loose-file version wrote
    arcname = f"audio/{rid[:2]}/{rid}.flac"
    buf = io.BytesIO()
    sf.write(buf, wav, SR, format="FLAC", subtype="PCM_16")
    shard_zip(shard_of(rid)).writestr(arcname, buf.getvalue())
    if src == "test":
        split = "test"
    elif not core:
        # multilingual tail is train-only on purpose: val must stay EN+HI so
        # best-checkpoint selection is comparable across every experiment.
        split = "train"
    else:
        split = "val" if int(hashlib.md5(rid.encode()).hexdigest(), 16) % 100 < VAL_PCT else "train"
    rec = {
        "id": rid, "path": arcname, "label": label,
        "language": lang,
        "midfiller": None if row["midfiller"] is None else bool(row["midfiller"]),
        "endfiller": None if row["endfiller"] is None else bool(row["endfiller"]),
        "synthetic": bool(row["synthetic"]),
        "dataset": row.get("dataset"), "duration_s": round(len(wav) / SR, 3),
        "split": split, "source": "real", "kind": "",
    }
    mf.write(json.dumps(rec) + "\n")
    done_ids.add(rid)
    counts[(lang, label, src)] = counts.get((lang, label, src), 0) + 1
    return 1

try:
    for src, name in [("train", "pipecat-ai/smart-turn-data-v3.2-train"),
                      ("test", "pipecat-ai/smart-turn-data-v3.2-test")]:
        dd = load_dataset(name, streaming=True)
        split_name = "train" if "train" in dd else list(dd.keys())[0]
        ds = dd[split_name].cast_column("audio", Audio(sampling_rate=SR))
        t0, n_scanned, n_kept = time.time(), 0, 0
        for row in ds:
            n_scanned += 1
            n_kept += process(row, src)
            if n_scanned % 5000 == 0:
                mf.flush()
                print(f"[{src}] scanned {n_scanned} kept {n_kept} "
                      f"({(time.time()-t0)/60:.1f} min, {len(zips)} shards open)",
                      flush=True)
        mf.flush()
        print(f"[{src}] DONE: scanned {n_scanned}, kept {n_kept}")
finally:
    mf.close()
    # closing is what writes each shard's central directory -- without it the
    # zips are unreadable, so this must happen even when the stream blows up
    for zf in zips.values():
        zf.close()
    print(f"closed {len(zips)} shard zips")

In [ ]:
import json, zipfile
import polars as pl
from pathlib import Path

OUT = Path("/kaggle/working/prep")
seen, unique, bad = set(), [], 0
for line in open(OUT / "manifest.jsonl"):
    try:
        r = json.loads(line)
    except json.JSONDecodeError:
        bad += 1
        continue
    if r["id"] not in seen:
        seen.add(r["id"])
        unique.append(r)
if bad:
    print(f"skipped {bad} unparseable manifest lines")
# infer_schema_length=None: midfiller/endfiller are null for long stretches
df = pl.DataFrame(unique, infer_schema_length=None)
df.write_parquet(OUT / "manifest.parquet")

print(f"total {df.height} clips, {df['duration_s'].sum()/3600:.1f} h")
print(df.group_by(["split", "language", "label"]).len().sort(["split", "language", "label"]))
print(df.group_by("split").agg(
    pl.col("midfiller").mean().alias("midfiller_rate"),
    pl.col("synthetic").mean().alias("synthetic_rate"),
))
# ---- shard check: every zip must open, and together hold every manifest row.
# testzip() re-CRCs ~15 GB (far too slow); opening the archive already validates
# the central directory, which is the failure mode a killed run actually causes.
shards = sorted((OUT / "shards").glob("shard_*.zip"))
entries, sizes = 0, []
for p in shards:
    with zipfile.ZipFile(p) as zf:
        n = len(zf.namelist())
    entries += n
    sizes.append((p.stat().st_size / 1e9, n))
gb = sorted(s for s, _ in sizes)
print(f"\n{len(shards)} shards, {entries} entries, {sum(gb):.1f} GB")
if shards:
    print(f"per-shard GB: min {gb[0]:.2f} / median {gb[len(gb)//2]:.2f} / max {gb[-1]:.2f}"
          f"  |  entries: min {min(n for _, n in sizes)} / max {max(n for _, n in sizes)}")
# >= not ==: a crash-and-resume can leave a duplicate arcname for the clip that
# was mid-write, and extraction just keeps the last copy. Fewer entries than
# rows means audio is genuinely missing -> re-run prep, do not train on it.
assert entries >= df.height, f"{entries} shard entries < {df.height} manifest rows"

total_gb = sum(f.stat().st_size for f in OUT.rglob("*")) / 1e9
print(f"output size: {total_gb:.1f} GB (expect ~16-17 GB; must stay under ~19.6 GB)")
if total_gb > 19.0:
    print("!! too close to the /kaggle/working limit — lower OTHER_CAP_PER_LANG_LABEL")